# Workflow 3: Compare Structural Ensembles to High-Throughput Experimental Data

## Goal

Test whether specific metastable states from simulation are statistically consistent with experimental conformational signals (LiP-MS and XL-MS). Identify and inspect representative structures from the best-supported metastable states.

---

## Typical runtime

| Step | Runtime |
|------|----------|
| Experimental input check | < 1 minute |
| Verify Workflow 2 outputs | < 1 minute |
| **LiP-MS / XL-MS consistency test** | **1–3 hours** (full production data) |
| Inspect one representative example | < 1 minute |

> **Note:** `MassSpec` now collects SASA and XP inputs automatically from the Workflow 2 output directories when cached arrays are not already present. This notebook follows that current workflow rather than requiring pre-built `SASA.npy` or `Jwalk.npy` files up front.

---

## Required input files

| File | Path | Role |
|------|------|------|
| LiP-MS experimental data | `$DATASTORE/user_input/experimental_data/ecPGK_significant_LiPMS_peptide_R1_merged.xlsx` | Experimental signals |
| XL-MS experimental data | `$DATASTORE/user_input/experimental_data/ecPGK_significant_XLMS_peptide_R1_merged.xlsx` | Cross-link data |
| MSM mapping (from Workflow 2) | `$DATASTORE/outputs/workflow2/MSM/1ZMR_prod_MSMmapping.csv` | Metastable state assignments |
| MSM meta distribution | `$DATASTORE/outputs/workflow2/MSM/1ZMR_prod_meta_dist.npy` | Metastable-state probability distribution data |
| Per-trajectory SASA files | `$DATASTORE/outputs/workflow2/OP_AA/SASA/1ZMR_Traj{N}.SASA` | Workflow 2 SASA inputs for auto-collection |
| Per-trajectory XP files | `$DATASTORE/outputs/workflow2/OP_AA/XP/1ZMR_Traj{N}.XP` | Workflow 2 XP inputs for auto-collection or streaming |
| Workflow 2 OP directory | `$DATASTORE/outputs/workflow2/OP_last335/` | Q and G values used when reporting representatives |
| Cluster data | `$DATASTORE/outputs/workflow2/nonnative_clustering_last335/cluster_data_topoly_linking_number.npz` | Entanglement-change classes |
| Native all-atom PDB | `$DATASTORE/user_input/reference_structures/1zmr_model_clean.pdb` | Reference structure |
| AA trajectory directory | `$DATASTORE/user_input/aa_trajectories/` | Source trajectories for representative structures |

> **Environment:** Before running this notebook, activate the `ncledetector` conda environment and register it as a kernel:
```bash
conda activate ncledetector
python -m ipykernel install --user --name ncledetector --display-name "ncledetector"
```
Then select **ncledetector** from the kernel picker (top-right of VS Code or Jupyter).

## Step 1. Set up paths

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

DATASTORE = "/scratch/ims86/NCLEdetector_Datastore"
OUTDIR    = f"{DATASTORE}/outputs/workflow3"

# Create output directory if needed
os.makedirs(f"{OUTDIR}/MassSpec_ConsistencyTest", exist_ok=True)

print(f"DATASTORE: {DATASTORE}")
print(f"OUTDIR:    {OUTDIR}")
print("\n✓ Directories ready")

## Step 2. Verify experimental input files

Load and display the processed experimental data. These files should contain LiP-MS and XL-MS signals that were previously mapped to residue-level significance.

In [ ]:
# Load experimental data
lipms_file = f"{DATASTORE}/user_input/experimental_data/ecPGK_significant_LiPMS_peptide_R1_merged.xlsx"
xlms_file  = f"{DATASTORE}/user_input/experimental_data/ecPGK_significant_XLMS_peptide_R1_merged.xlsx"

print("Loading experimental files...\n")

lipms = pd.read_excel(lipms_file)
xlms  = pd.read_excel(xlms_file)

print(f"LiP-MS data shape: {lipms.shape}")
print(f"Columns: {list(lipms.columns)}")
print(lipms.head())

print(f"\nXL-MS data shape: {xlms.shape}")
print(f"Columns: {list(xlms.columns)}")
print(xlms.head())

print(f"\n✓ Experimental files loaded successfully")

## Step 3. Verify Workflow 2 SASA, XP, and trajectory inputs

Check that the per-trajectory SASA and XP files from Workflow 2 are present. Cached arrays in `workflow3/` are optional because `MassSpec` can build them automatically when needed.

In [ ]:
sasa_dir = Path(f"{DATASTORE}/outputs/workflow2/OP_AA/SASA")
xp_dir = Path(f"{DATASTORE}/outputs/workflow2/OP_AA/XP")
op_last335_dir = Path(f"{DATASTORE}/outputs/workflow2/OP_last335")
cluster_data_file = Path(f"{DATASTORE}/outputs/workflow2/nonnative_clustering_last335/cluster_data_topoly_linking_number.npz")
native_aa_pdb = Path(f"{DATASTORE}/user_input/reference_structures/1zmr_model_clean.pdb")
aa_traj_dir = Path(f"{DATASTORE}/user_input/aa_trajectories")

print("Checking Workflow 2 inputs...\n")
for path in [sasa_dir, xp_dir, op_last335_dir, cluster_data_file, native_aa_pdb, aa_traj_dir]:
    print(f"{path}: {path.exists()}")

sasa_files = sorted(sasa_dir.glob("*.SASA"))
xp_files = sorted(xp_dir.glob("*.XP"))
print(f"\nSASA files found: {len(sasa_files)}")
print("SASA sample:", [p.name for p in sasa_files[:5]])
print(f"\nXP files found: {len(xp_files)}")
print("XP sample:", [p.name for p in xp_files[:5]])

cached_sasa = Path(f"{OUTDIR}/SASA.npy")
cached_jwalk = Path(f"{OUTDIR}/Jwalk.npy")
print(f"\nCached SASA array exists: {cached_sasa.exists()}")
print(f"Cached Jwalk array exists: {cached_jwalk.exists()}")
if not cached_sasa.exists():
    print("SASA.npy is not cached yet. MassSpec will build it automatically from the per-trajectory SASA files.")
if not cached_jwalk.exists():
    print("Jwalk.npy is not cached yet. This is optional because XP files can be streamed directly during analysis.")

## Step 4. Load MSM data from Workflow 2

Load the MSM mapping and metastable state definitions that were created in Workflow 2.

In [ ]:
print("Loading Workflow 2 MSM outputs...\n")

msm_mapping_file = f"{DATASTORE}/outputs/workflow2/MSM/1ZMR_prod_MSMmapping.csv"
meta_dist_file = f"{DATASTORE}/outputs/workflow2/MSM/1ZMR_prod_meta_dist.npy"
meta_set_file = f"{DATASTORE}/outputs/workflow2/MSM/1ZMR_prod_meta_set.csv"

# Load MSM mapping
msm_mapping = pd.read_csv(msm_mapping_file)
print(f"MSM mapping shape: {msm_mapping.shape}")
print(f"Columns: {list(msm_mapping.columns)}")
print(msm_mapping.head())

# Check metastable states present
meta_states = sorted(msm_mapping['metastablestate'].unique())
print(f"\nMetastable states found: {meta_states}")

# Load meta distribution
meta_dist = np.load(meta_dist_file)
print(f"\nMeta distribution shape: {meta_dist.shape}")
print(f"State probabilities: {meta_dist}")

print(f"\n✓ MSM data loaded successfully")

## Step 5. Initialize `MassSpec` for the consistency test

⚠️ **WARNING: This consistency test runs on the full tutorial ensemble and can take 1–3 hours.**

`MassSpec` is configured here to auto-collect SASA data from Workflow 2 outputs if a cached `SASA.npy` file is not already present. XP data can be streamed directly from the Workflow 2 `.XP` files.

In [ ]:
from NCLEdetector.compare_sim2exp import MassSpec

print("="*80)
print("INITIALIZING WORKFLOW 3 CONSISTENCY TEST")
print("="*80)
print("\n⏱️  EXPECTED RUNTIME: 1–3 hours (full production ensemble)")
print("\n⚠️  This analysis uses the full Workflow 2 cohort. Do not interrupt the run once started.\n")

PROT_LEN = 387
N_TRAJ = 1000
SASA_XP_FRAMES_PER_TRAJ = 67
N_ANALYSIS_FRAMES = 67
STATE_IDX_LIST = [4, 6, 8]
NATIVE_STATE_IDX = 9
RM_TRAJ_LIST = [65, 75, 155, 162, 199, 231, 264, 286, 296, 314, 354, 417, 448, 472, 473, 474, 577, 579, 591, 703, 704, 732, 758, 812, 833, 870, 876, 944, 967]

AA_TRAJ_DIR = f"{DATASTORE}/user_input/aa_trajectories"
NATIVE_AA_PDB = f"{DATASTORE}/user_input/reference_structures/1zmr_model_clean.pdb"
CLUSTER_DATA_FILE = f"{DATASTORE}/outputs/workflow2/nonnative_clustering_last335/cluster_data_topoly_linking_number.npz"
OPPATH = f"{DATASTORE}/outputs/workflow2/OP_last335/"
SASA_DIR = f"{DATASTORE}/outputs/workflow2/OP_AA/SASA"
XP_DIR = f"{DATASTORE}/outputs/workflow2/OP_AA/XP"

print("Configuration:")
print(f"  Trajectories: {N_TRAJ}")
print(f"  Frames per trajectory: {SASA_XP_FRAMES_PER_TRAJ}")
print(f"  Last frames analyzed per trajectory: {N_ANALYSIS_FRAMES}")
print(f"  Protein length: {PROT_LEN} residues")
print(f"  States to test: {STATE_IDX_LIST}")
print(f"  Native state index: {NATIVE_STATE_IDX}")
print(f"  Mirror trajectories removed: {len(RM_TRAJ_LIST)}")
print("\nInitializing MassSpec object...")

try:
    MS = MassSpec(
        msm_data_file=msm_mapping_file,
        meta_dist_file=meta_dist_file,
        LiPMS_exp_file=lipms_file,
        XLMS_exp_file=xlms_file,
        cluster_data_file=CLUSTER_DATA_FILE,
        OPpath=OPPATH,
        AAdcd_dir=AA_TRAJ_DIR,
        native_AA_pdb=NATIVE_AA_PDB,
        sasa_dir=SASA_DIR,
        xp_dir=XP_DIR,
        n_traj=N_TRAJ,
        sasa_xp_frames_per_traj=SASA_XP_FRAMES_PER_TRAJ,
        state_idx_list=STATE_IDX_LIST,
        prot_len=PROT_LEN,
        n_analysis_frames=N_ANALYSIS_FRAMES,
        rm_traj_list=RM_TRAJ_LIST,
        native_state_idx=NATIVE_STATE_IDX,
        outdir=f"{OUTDIR}/MassSpec_ConsistencyTest",
        ID="1ZMR",
        start = 6332,
        end=-1,
        stride=1,
        num_perm=1000,
        n_boot=100,
        lag_frame=20,
        nproc=8,
    )
    print("✓ MassSpec initialized successfully")
except Exception as e:
    print(f"✗ Error initializing MassSpec: {e}")
    import traceback
    traceback.print_exc()

## Step 6. RUN CONSISTENCY TEST (Full Production Data)

This cell runs the actual consistency test. **PLEASE BE PATIENT—this will take 1–3 hours.**

In [ ]:
import time
from datetime import datetime, timedelta

print("\n" + "="*80)
print("STARTING CONSISTENCY TEST")
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)
print("\nThis will evaluate:")
print(f"  • {len(STATE_IDX_LIST)} metastable states")
print(f"  • {N_TRAJ} trajectories × {SASA_XP_FRAMES_PER_TRAJ} frames each = {N_TRAJ * SASA_XP_FRAMES_PER_TRAJ:,} sampled frames")
print("  • LiP-MS and XL-MS consistency signals")
print(f"\nEstimated completion time: {datetime.now() + timedelta(hours=2)}\n")

start_time = time.time()

try:
    consist_data_file, consist_result_file = MS.LiP_XL_MS_ConsistencyTest()
    elapsed = time.time() - start_time

    print("\n" + "="*80)
    print("✓ CONSISTENCY TEST COMPLETED")
    print("="*80)
    print(f"Elapsed time: {elapsed/60:.1f} minutes ({elapsed/3600:.2f} hours)")
    print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("\nOutput files:")
    print(f"  Consistency data: {consist_data_file}")
    print(f"  Results workbook: {consist_result_file}")

except Exception as e:
    elapsed = time.time() - start_time
    print(f"\n✗ ERROR after {elapsed/60:.1f} minutes:")
    print(f"  {e}")
    import traceback
    traceback.print_exc()

## Step 7. Inspect one representative-structure example

Generating the full representative-structure set can be expensive and can produce a large number of outputs. In this notebook, stop after inspecting one example signal in one state rather than calling `MS.select_rep_structs()` for the full result set.

If a previously generated `Consistent_structures_v8.xlsx` workbook is available, the next cell loads one example row from a single state sheet and reports the trajectory and frame to inspect manually.

In [ ]:
rep_file = Path(f"{OUTDIR}/MassSpec_ConsistencyTest/Consistent_structures_v8.xlsx")
example_sheet = "State 5"

if rep_file.exists():
    rep_df = pd.read_excel(rep_file, sheet_name=example_sheet).dropna(how="all")
    if rep_df.empty:
        print(f"Workbook found, but {example_sheet} has no rows.")
    else:
        example_row = rep_df.iloc[0]
        print(f"Representative workbook: {rep_file}")
        print(f"Example state sheet: {example_sheet}")
        print("\nOne example signal in one state:")
        for column in rep_df.columns:
            print(f"  {column}: {example_row[column]}")

        print("\nManual inspection hint:")
        print("  Use the reported trajectory/frame pair with the corresponding AA trajectory in VMD or PyMOL.")
else:
    print("Representative-structure generation is intentionally skipped in this notebook.")
    print("If you need the full workbook and extracted structures, run this manually outside the notebook example:")
    print("MS.select_rep_structs(consist_data_file, consist_result_file, total_traj_num_frames=335, n_analysis_frames=67)")

## Step 8. Inspect consistency test results

Load the LiP-MS and XL-MS workbook sheets and review the strongest signals from the tested metastable states.

In [ ]:
import glob

outdir = Path(f"{OUTDIR}/MassSpec_ConsistencyTest")
pvalue_files = sorted(outdir.glob("LiPMS_XLMS_consist_pvalues_metastates_*.xlsx"))

def clean_consistency_sheet(df):
    signal_col = df.columns[0]
    df = df.rename(columns={signal_col: "signal"}).dropna(how="all")
    df = df[df["signal"].notna()]
    df = df[df["Near-native state"].notna()]
    return df

if pvalue_files:
    pvalue_file = pvalue_files[-1]
    print(f"Loading workbook: {pvalue_file}\n")

    lipms_df = clean_consistency_sheet(pd.read_excel(pvalue_file, sheet_name="LiPMS"))
    xlms_df = clean_consistency_sheet(pd.read_excel(pvalue_file, sheet_name="XLMS"))

    print("Top LiP-MS rows:")
    print(lipms_df.head(10))

    print("\nTop XL-MS rows:")
    print(xlms_df.head(10))
else:
    print(f"No consistency workbook found in {outdir}")
    print("Run Step 6 first.")

## Step 9. Summary and next steps

Review the consistency test results and prepare for downstream analysis.

In [ ]:
print("="*80)
print("WORKFLOW 3 SUMMARY")
print("="*80)

outdir = f"{OUTDIR}/MassSpec_ConsistencyTest"

print(f"\nOutput files generated in: {outdir}")
print("\nKey results:")
print("  1. LiP-MS / XL-MS consistency workbook across tested metastable states")
print("  2. Raw consistency arrays saved in the NPZ archive")
print("  3. One representative-structure example can be inspected from an existing workbook, if available")

print("\nNext steps:")
print("  1. Review the LiPMS and XLMS workbook sheets for significant state-signal pairs")
print("  2. If needed, generate the full representative-structure workbook offline with MS.select_rep_structs(...)")
print("  3. Load the reported trajectory/frame example into VMD or PyMOL for structural inspection")

print("\n✓ Workflow 3 notebook review complete!")